In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import matplotlib.dates as mdates
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

In [2]:
data = "/content/Data_Lion.xlsx - Sheet3.csv"
df = pd.read_csv(data, thousands=',')


df = df.sort_values("Date").reset_index(drop=True)

print(f"Dataset loader: {len(df)} quaraters of data")
display(df)

Dataset loader: 42 quaraters of data


,Date,Revenue,Operating Income,Net Profit,COGS,Pharmaceuticals_Cosmetics,Chemicals,Drug Store,Everyday Items Exp,CPI_YoY,Exchange_Rate
0,2014/01/01-2014/03/31,89596,3616,2347,39114,3029.33,8063.33,415734.0000,29500,-0.7,95.09
1,2014/04/01-2014/06/30,87037,342,119,38115,2848.33,7579.33,389044.6667,29500,-0.3,92.24
2,2014/07/01-2014/09/30,88672,3385,2429,38863,3059.00,7747.67,416080.0000,29500,0.2,96.27
3,2014/10/01-2014/12/31,102091,5063,2473,44664,3380.33,7941.00,424973.3333,29500,0.7,94.38
4,2015/01/01-2015/03/31,82925,2137,1136,36654,2948.00,7450.67,417344.0000,31624,0.9,96.08
5,2015/04/01-2015/06/30,97358,3561,2762,42192,3226.00,7298.33,446027.3333,31624,1.1,94.58
6,2015/07/01-2015/09/30,93744,5372,3515,39998,4990.67,7710.33,457529.6667,31624,1.1,95.01
7,2015/10/01-2015/12/31,104632,5304,3267,43666,5503.67,7525.67,466065.3333,31624,1.5,93.35
8,2016/01/01-2016/03/31,89485,5795,3797,36914,4452.67,7134.33,456245.3333,33192,1.6,90.20
9,2016/04/01-2016/06/30,98763,4635,3364,40546,4838.67,6771.33,476551.3333,33192,1.4,90.17


In [3]:
df["Date"] = df["Date"].str.split("-").str[1]
df["Date"] = pd.to_datetime(df["Date"])

In [4]:
df = df.sort_values("Date").reset_index(drop=True)

def create_features(data):
  df_feat = data.copy()

  # Feature 1: Seasonality (Quarter 1-4)
  # Divide Quarters
  df_feat["Quarter"] = df_feat["Date"].dt.quarter

  # Feature 2: Lags (Past Performance)
  # The best predictor of tomorrow is often todat (Lag 1) or this time last year
  for col in ["Revenue", "Operating Income", "Net Profit"]:
    df_feat[f"{col}_Lag1"] = df_feat[col].shift(1) # Previous Quarter
    df_feat[f"{col}_Lag4"] = df_feat[col].shift(4) # Same Quarter Last Year

  # B. MACRO LAGS (Your Request: "Known at time t")
    # We shift these so we predict T using data from T-1
  macros = ['Pharmaceuticals_Cosmetics', 'Chemicals', 'CPI_YoY', 'Exchange_Rate', "Drug Store", "Everyday Items Exp", "COGS"]
  for col in macros:
      df_feat[f'{col}_Lag1'] = df_feat[col].shift(1)

  # Drop the first 4 rows because they now have NaNs (no history for them)
  df_feat = df_feat.dropna().reset_index(drop=True)

  return df_feat

df_model = create_features(df)


In [5]:
features_rev = ["Quarter", "Pharmaceuticals_Cosmetics_Lag1", "Chemicals_Lag1", "Drug Store_Lag1", "Everyday Items Exp_Lag1", "Revenue_Lag1", "Revenue_Lag4", "CPI_YoY_Lag1", "Exchange_Rate_Lag1"]
target_rev = "Revenue"

In [6]:
X_rev = df_model[features_rev]
y_rev = df_model[target_rev]

In [7]:
model_rev = RandomForestRegressor(
        n_estimators=200, random_state=42
    )

model_rev.fit(X_rev, y_rev)

RandomForestRegressor(n_estimators=200, random_state=42)

In [8]:
features_op = [
    "Quarter",
    "Pharmaceuticals_Cosmetics_Lag1",
    "Chemicals_Lag1",
    "Drug Store_Lag1",
    "Everyday Items Exp_Lag1",
    "Operating Income_Lag1",
    "Operating Income_Lag4",
    "CPI_YoY_Lag1",
    "Exchange_Rate_Lag1",
    "COGS_Lag1",
    "Revenue"  # <--- The Critical Feature
]
target_op = "Operating Income"

In [9]:
X_op = df_model[features_op]
y_op = df_model[target_op]

In [10]:
model_op = RandomForestRegressor(
        n_estimators=200, random_state=42
    )

model_op.fit(X_op, y_op)

RandomForestRegressor(n_estimators=200, random_state=42)

In [11]:
features_np = [
    "Quarter",
    "Pharmaceuticals_Cosmetics_Lag1",
    "Chemicals_Lag1",
    "Drug Store_Lag1",
    "Everyday Items Exp_Lag1",
    "Net Profit_Lag1",
    "Net Profit_Lag4",
    "CPI_YoY_Lag1",
    "Exchange_Rate_Lag1",
    "COGS_Lag1",
    "Revenue",           # <--- Chain Link 1
    "Operating Income"   # <--- Chain Link 2
]
target_np = "Net Profit"

X_np = df_model[features_np]
y_np = df_model[target_np]

In [12]:
model_np = RandomForestRegressor(
        n_estimators=200, random_state=42
    )

model_np.fit(X_np, y_np)

RandomForestRegressor(n_estimators=200, random_state=42)

In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb

# [Assuming 'df' is your current dataframe ending in June 2024]
# [Assuming models 'model_rev', 'model_op', 'model_np' are already trained on available data]

# 1. Define Simulation Parameters
# Last Actual Data Point = Q2 2024 (approx April 1st or June 30th depending on your file)
# We find it dynamically:
last_known_date = df['Date'].max()
target_date = pd.Timestamp("2026-10-01")    # The goal: Q4 2026

# Create History Buffer
history = df.copy()

print(f"--- STARTING MULTI-STEP FORECAST ---")
print(f"From: {last_known_date.date()} (Last Actual)")
print(f"To:   {target_date.date()} (Target)\n")
print(f"{'Date':<12} | {'Revenue':>12} | {'Op Income':>12} | {'Net Profit':>12}")
print("-" * 60)

# 2. The Loop
# We advance 3 months from the last known date to start the first prediction
current_date = last_known_date + pd.DateOffset(months=3)

while current_date <= target_date:
    # A. Get Lags from the growing 'history' dataframe
    row_lag1 = history.iloc[-1] # Previous Quarter (could be a prediction)
    row_lag4 = history.iloc[-4] # Previous Year (could be actual or prediction)

    # B. Construct Inputs
    input_row = {
        'Quarter': (current_date.month // 3) % 4 + 1,

        # --- MACRO LAGS (Carrying forward last known actuals) ---
        'Pharmaceuticals_Cosmetics_Lag1': row_lag1['Pharmaceuticals_Cosmetics'],
        'Chemicals_Lag1': row_lag1['Chemicals'],
        'CPI_YoY_Lag1': row_lag1['CPI_YoY'],
        'Exchange_Rate_Lag1': row_lag1['Exchange_Rate'],
        'Drug Store_Lag1': row_lag1['Drug Store'],
        'Everyday Items Exp_Lag1': row_lag1['Everyday Items Exp'],
        'COGS_Lag1': row_lag1['COGS'],

        # --- PERFORMANCE LAGS ---
        'Revenue_Lag1': row_lag1['Revenue'],
        'Revenue_Lag4': row_lag4['Revenue'],
        'Operating Income_Lag1': row_lag1['Operating Income'],
        'Operating Income_Lag4': row_lag4['Operating Income'],
        'Net Profit_Lag1': row_lag1['Net Profit'],
        'Net Profit_Lag4': row_lag4['Net Profit']
    }

    # C. Predict REVENUE
    # (Ensure the column order matches your training data exactly)
    input_df_rev = pd.DataFrame([input_row])[features_rev]
    pred_revenue = model_rev.predict(input_df_rev)[0]

    # D. Predict OP INCOME (Inject Revenue)
    input_row['Revenue'] = pred_revenue
    input_df_op = pd.DataFrame([input_row])[features_op]
    pred_op = model_op.predict(input_df_op)[0]

    # E. Predict NET PROFIT (Inject Op Income)
    input_row['Operating Income'] = pred_op
    input_df_np = pd.DataFrame([input_row])[features_np]
    pred_np = model_np.predict(input_df_np)[0]

    # F. Print & Check Targets
    if current_date == pd.Timestamp('2025-10-01') or current_date == pd.Timestamp('2026-10-01'):
         print(f"*** TARGET Q4 {current_date.year} ***")

    print(f"{current_date.date()} | {pred_revenue:,.0f} | {pred_op:,.0f} | {pred_np:,.0f}")

    # G. Append to History
    new_row = {
        'Date': current_date,
        'Revenue': pred_revenue,
        'Operating Income': pred_op,
        'Net Profit': pred_np,

        # IMPORTANT: Carry forward the macro vars so they exist for the next loop's Lag1
        'Pharmaceuticals_Cosmetics': row_lag1['Pharmaceuticals_Cosmetics'],
        'Chemicals': row_lag1['Chemicals'],
        'CPI_YoY': row_lag1['CPI_YoY'],
        'Exchange_Rate': row_lag1['Exchange_Rate'],
        'Drug Store': row_lag1['Drug Store'],
        'Everyday Items Exp': row_lag1['Everyday Items Exp'],
        'COGS': row_lag1['COGS']
    }

    history = pd.concat([history, pd.DataFrame([new_row])], ignore_index=True)

    current_date = current_date + pd.DateOffset(months=3)

print("-" * 60)
print("Forecast Complete.")

--- STARTING MULTI-STEP FORECAST ---
From: 2024-06-30 (Last Actual)
To:   2026-10-01 (Target)

Date         |      Revenue |    Op Income |   Net Profit
------------------------------------------------------------
2024-09-30 | 106,696 | 6,208 | 4,480
2024-12-30 | 103,650 | 5,724 | 3,627
2025-03-30 | 98,075 | 5,584 | 3,646
2025-06-30 | 105,912 | 6,799 | 4,765
2025-09-30 | 106,569 | 6,609 | 4,765
2025-12-30 | 104,042 | 5,608 | 3,621
2026-03-30 | 103,182 | 5,849 | 3,679
2026-06-30 | 106,603 | 6,463 | 4,755
2026-09-30 | 106,569 | 6,217 | 4,447
------------------------------------------------------------
Forecast Complete.


In [15]:
# ==========================================
# CALCULATION: AGGREGATE FY FROM 'HISTORY'
# ==========================================

# 1. Extract Year from the Date column in your 'history' dataframe
# (Make sure 'history' is the dataframe you used in the loop)
history['Year'] = pd.to_datetime(history['Date']).dt.year

# 2. Filter for only the forecast years (2025 and 2026)
df_fy_forecast = history[history['Year'].isin([2025, 2026])].copy()

# 3. Group by Year and Sum the quarterly values
df_fy_totals = df_fy_forecast.groupby('Year')[['Revenue', 'Operating Income', 'Net Profit']].sum().reset_index()

# 4. Display the Final Table
print("\n--- FINAL FULL YEAR (FY) PROJECTIONS (Millions JPY) ---")
print(df_fy_totals.to_string(index=False, float_format="{:,.0f}".format))

# 5. Calculate & Print Growth Rates
if len(df_fy_totals) == 2:
    p_2025 = df_fy_totals[df_fy_totals["Year"] == 2025].iloc[0]
    p_2026 = df_fy_totals[df_fy_totals["Year"] == 2026].iloc[0]

    print("\n--- YoY Growth (2026 vs 2025) ---")
    for col in ["Revenue", "Operating Income", "Net Profit"]:
        val_25 = p_2025[col]
        val_26 = p_2026[col]
        growth = ((val_26 - val_25) / val_25) * 100
        print(f"{col:<20} : {growth:>6.2f}%")


--- FINAL FULL YEAR (FY) PROJECTIONS (Millions JPY) ---
 Year  Revenue  Operating Income  Net Profit
 2025  414,598            24,600      16,796
 2026  316,354            18,529      12,880

--- YoY Growth (2026 vs 2025) ---
Revenue              : -23.70%
Operating Income     : -24.68%
Net Profit           : -23.31%
